# Universal High-Fidelity Binarization & Adaptive Noise Reduction

## 1. Objective
This notebook executes a robust, hardware-negotiated binarization pipeline designed to process scanned financial PDFs of varying qualities. It dynamically scales the rendering resolution to align with system hardware limits, uses edge-preserving bilateral filtering to suppress scanner noise, and computes the absolute mathematically optimal threshold for each page using Otsu's Binarization—delivering a flawless, high-contrast 1-bit binary canvas (`.tiff`) optimized for OCR.

---

## 2. Technical Challenges & Elegant Solutions

### Challenge A: Memory Exhaustion (OOM) at Ultra-High Resolutions
To capture the finest details of thin Arabic fonts, signatures, and decimal points, we target a high-density page render ($5.0\times$ zoom, equivalent to $\approx 360\text{ DPI}$). However, generating such massive canvases can deplete system RAM on limited environments, resulting in hard Out-Of-Memory (OOM) crashes.
* **Solution (Hardware-Negotiated Step-Down Backoff Loop):** 
  The pipeline wraps the rendering logic in a dynamic trial-and-error loop. It first requests the ideal $5.0\times$ scale. If the system memory manager throws an allocation error, the code catches the exception, reduces the scale by $0.5\times$ steps, and retries. If critical limits are breached, it enforces a $2.0\times$ emergency fallback—ensuring the script never crashes and always extracts the highest quality the hardware can afford.

### Challenge B: Scanner Noise, Fold Shadows, and Speckle Artifacts
Scanned documents are rarely pristine; they often contain paper textures, dust specks, and non-uniform scanner lighting. Applying traditional Gaussian blurs to eliminate this noise is destructive, as it averages pixels indiscriminately, smudging the sharp edges of characters and rendering them unreadable for OCR engines.
* **Solution (Edge-Preserving Bilateral Filtering):**
  The pipeline employs a Bilateral Filter (`cv2.bilateralFilter`) that operates in both the spatial and radiometric (color intensity) domains. It smooths out flat, low-frequency regions (paper texture and dirt) while strictly preserving sharp, high-contrast boundaries (ink-to-paper transitions):
  $$\sigma_{\text{space}} = 75, \quad \sigma_{\text{color}} = 75$$
  This leaves the background perfectly flat and clean while maintaining razor-sharp text outlines.

### Challenge C: Faded Ink vs. Shadows (Static Thresholding Failures)
Scanned documents have inconsistent exposures. A static threshold (e.g., $200$) fails catastrophically on dark/shadowed scans (turning entire sections pitch black) or on carbon copies with faded ink (making the text completely disappear into the white background).
* **Solution (Mathematical Optimization via Otsu's Binarization):**
  Instead of relying on fixed values, the pipeline dynamically calculates the ideal threshold $t$ by modeling the image histogram as a bimodal distribution (separating background paper from foreground ink). It searches for the threshold that minimizes the **within-class variance** $\sigma^2_w(t)$:
  $$\sigma^2_w(t) = \omega_0(t)\sigma^2_0(t) + \omega_1(t)\sigma^2_1(t)$$
  This mathematical split ensures that the background is swept to pure white ($255$) and the text is pushed to solid black ($0$), regardless of whether the original scan was over-exposed or heavily shadowed.

In [ ]:
import fitz  
import cv2
import numpy as np
import os

print("[+] Starting Universal PDF Pre-processing Pipeline...")

pdf_name = "كشف حساب الراجحي.pdf"

if not os.path.exists(pdf_name):
    print(f"[!] Error: Cannot find '{pdf_name}' in the current folder!")
else:
    doc = fitz.open(pdf_name)
    page = doc.load_page(0)

    try:
        fitz.tools.set_aa_level(0)
    except Exception:
        pass

    optimal_zoom = 5.0
    pix = None

    while optimal_zoom >= 3.0:
        matrix = fitz.Matrix(optimal_zoom, optimal_zoom)
        try:
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            break
        except Exception:
            print(f"[!] RAM allocation failed at {optimal_zoom}x zoom. Stepping down...")
            optimal_zoom -= 0.5

    if pix is None:
        print("[!] Emergency fallback to 2.0x zoom.")
        try:
            matrix = fitz.Matrix(2.0, 2.0)
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            optimal_zoom = 2.0
        except Exception as e:
            print(f"[!] Critical Error! {e}")

    if pix is not None:
        print(f"[✓] Canvas successfully rendered at {optimal_zoom:.1f}x zoom.")

        img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape((pix.h, pix.w, pix.n))
        img_gray = cv2.cvtColor(img_data, cv2.COLOR_RGB2GRAY)

        print("[+] Applying Bilateral Filter (Edge-Preserving Smoothness)...")
        img_filtered = cv2.bilateralFilter(img_gray, d=9, sigmaColor=75, sigmaSpace=75)

        print("[+] Calculating mathematically optimal threshold via Otsu's Binarization...")
        optimal_thresh, binary_1bit = cv2.threshold(
            img_filtered, 
            0, 
            255, 
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        print(f"[✓] Optimal dynamic threshold calculated: {optimal_thresh}")

        output_path = "page_1bit_universal.tiff"
        cv2.imwrite(output_path, binary_1bit)
        
        print(f"[✓] Flawless Universal File generated and saved to: {output_path}")